# Statistical Validation of A/B Testing Experiments for E-Commerce User Interface Optimization

**Domain:** E-Commerce & Supply Chain | **COs:** CO2, CO5 | **Level:** K4 (Analyze) | **Stack:** Python (SciPy Stats)

## Problem statement
An e-commerce company redesigns its product/checkout page (**Treatment**) and wants to know whether it truly performs better than the current page (**Control**), or whether the observed difference is just random noise.

## What this notebook does
1. Generates a realistic A/B test dataset (and shows how to plug in a real one)
2. Designs the experiment: hypotheses, sample-size / power analysis
3. Validates the experiment itself: Sample Ratio Mismatch (SRM), A/A test, peeking problem
4. Tests the **primary metric** (conversion rate): z-test, chi-square, Fisher's exact, confidence intervals
5. Tests **secondary metrics** (session duration, revenue per user, bounce rate): normality checks, Welch t-test, Mann-Whitney U, bootstrap
6. Handles **multiple comparisons** (Bonferroni, Holm, Benjamini-Hochberg) and **segment analysis**
7. Adds a Bayesian view and checks for a novelty effect
8. Ends with a data-driven launch decision

In [ ]:
# 1. SETUP
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (9, 5)
ALPHA = 0.05
print("NumPy", np.__version__, "| Pandas", pd.__version__)
import scipy; print("SciPy", scipy.__version__)

## 2. Data
We simulate a 14-day experiment where the new UI has a **real but modest effect**, mostly on mobile. Metrics per user:

| Column | Meaning |
|---|---|
| `group` | control / treatment |
| `device` | mobile / desktop / tablet |
| `converted` | 1 if the user purchased (primary metric) |
| `revenue` | order value (0 for non-buyers) |
| `session_sec` | session duration in seconds |
| `bounced` | 1 if user left after one page (guardrail metric) |
| `date` | day of the experiment |

In [ ]:
def generate_ab_data(n_control=10000, n_treatment=10000, seed=42):
    rng = np.random.default_rng(seed)
    n = n_control + n_treatment
    group = np.array(["control"] * n_control + ["treatment"] * n_treatment)
    is_t = (group == "treatment")

    device = rng.choice(["mobile", "desktop", "tablet"], size=n, p=[0.55, 0.35, 0.10])

    # baseline conversion by device + treatment uplift (strongest on mobile)
    base = pd.Series(device).map({"mobile": 0.09, "desktop": 0.13, "tablet": 0.10}).values
    lift = np.where(is_t, np.select([device == "mobile", device == "desktop"], [0.025, 0.008], 0.0), 0.0)
    converted = rng.binomial(1, base + lift)

    # revenue: log-normal order value for buyers only (heavy right tail, like real data)
    order_value = rng.lognormal(mean=4.0 + 0.03 * is_t, sigma=0.6, size=n)
    revenue = np.where(converted == 1, order_value, 0.0)

    # session duration (seconds): gamma distributed, slightly longer in treatment
    session_sec = rng.gamma(shape=2.0, scale=np.where(is_t, 82.5, 75.0))

    # bounce (guardrail): treatment slightly lower
    bounced = rng.binomial(1, np.where(is_t, 0.38, 0.42))

    date = pd.to_datetime("2026-09-01") + pd.to_timedelta(rng.integers(0, 14, size=n), unit="D")

    df = pd.DataFrame({
        "user_id": np.arange(1, n + 1),
        "group": group, "device": device, "date": date,
        "converted": converted, "revenue": revenue.round(2),
        "session_sec": session_sec.round(1), "bounced": bounced,
    })
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)

df = generate_ab_data()
df.head()

### (Optional) Use a real dataset instead
Popular public option: Kaggle **"A/B Testing" / `ab_data.csv`** (columns: `user_id, timestamp, group, landing_page, converted`).
Upload it via the Colab file panel, then uncomment and adapt the cell below. The rest of the notebook works as long as you have `group` (`control`/`treatment`) and `converted`; for other metrics, skip the sections whose columns are missing.

In [ ]:
# --- OPTIONAL: load real data (uncomment & adapt) ---
# raw = pd.read_csv("ab_data.csv")
# raw = raw[((raw.group == "control") & (raw.landing_page == "old_page")) |
#           ((raw.group == "treatment") & (raw.landing_page == "new_page"))]
# raw = raw.drop_duplicates("user_id")
# raw["date"] = pd.to_datetime(raw["timestamp"]).dt.normalize()
# df = raw[["user_id", "group", "date", "converted"]].copy()

## 3. Exploratory Data Analysis

In [ ]:
print(df.info())
print("\nMissing values:\n", df.isna().sum())
print("\nDuplicate users:", df.user_id.duplicated().sum())

summary = df.groupby("group").agg(
    users=("user_id", "count"),
    conversions=("converted", "sum"),
    conv_rate=("converted", "mean"),
    revenue_per_user=("revenue", "mean"),
    avg_session_sec=("session_sec", "mean"),
    bounce_rate=("bounced", "mean"),
).round(4)
summary

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))

sns.barplot(data=df, x="group", y="converted", ax=ax[0], errorbar=("ci", 95))
ax[0].set_title("Conversion rate (95% CI)")

sns.histplot(data=df, x="session_sec", hue="group", bins=50, element="step", ax=ax[1])
ax[1].set_title("Session duration (right-skewed)")

sns.histplot(data=df[df.revenue > 0], x="revenue", hue="group", bins=50, element="step", ax=ax[2])
ax[2].set_title("Order value (buyers only, heavy tail)")
plt.tight_layout(); plt.show()

## 4. Experiment design: hypotheses & power analysis

**Primary metric:** conversion rate (CR)

- H0: CR_treatment = CR_control (the new UI has no effect)
- H1: CR_treatment ≠ CR_control (two-sided)
- Significance level α = 0.05, desired power = 0.80

**Sample size per group** for a two-proportion test:

$$n = \frac{\left(z_{1-\alpha/2}\sqrt{2\bar p(1-\bar p)} + z_{1-\beta}\sqrt{p_1(1-p_1)+p_2(1-p_2)}\right)^2}{(p_2-p_1)^2}$$

In [ ]:
def sample_size_per_group(p1, mde_abs, alpha=0.05, power=0.80):
    p2 = p1 + mde_abs
    z_a = stats.norm.ppf(1 - alpha / 2)
    z_b = stats.norm.ppf(power)
    pbar = (p1 + p2) / 2
    n = (z_a * np.sqrt(2 * pbar * (1 - pbar)) + z_b * np.sqrt(p1 * (1 - p1) + p2 * (1 - p2))) ** 2 / mde_abs ** 2
    return int(np.ceil(n))

def power_two_prop(p1, p2, n, alpha=0.05):
    pbar = (p1 + p2) / 2
    se0 = np.sqrt(2 * pbar * (1 - pbar) / n)
    se1 = np.sqrt((p1 * (1 - p1) + p2 * (1 - p2)) / n)
    z_a = stats.norm.ppf(1 - alpha / 2)
    return 1 - stats.norm.cdf((z_a * se0 - abs(p2 - p1)) / se1)

p_base = df.loc[df.group == "control", "converted"].mean()
print(f"Baseline conversion: {p_base:.4f}")
for mde in [0.005, 0.01, 0.015, 0.02]:
    print(f"MDE = +{mde*100:.1f} pp  ->  required n per group = {sample_size_per_group(p_base, mde):,}")

ns = np.arange(500, 20001, 250)
plt.plot(ns, [power_two_prop(p_base, p_base + 0.015, n) for n in ns], label="MDE = 1.5 pp")
plt.plot(ns, [power_two_prop(p_base, p_base + 0.010, n) for n in ns], label="MDE = 1.0 pp")
plt.axhline(0.8, color="red", ls="--", label="80% power")
plt.xlabel("Users per group"); plt.ylabel("Power"); plt.title("Power curve"); plt.legend(); plt.show()

## 5. Validating the experiment itself (before looking at results)

### 5.1 Sample Ratio Mismatch (SRM)
If we intended a 50/50 split, the observed counts should not deviate significantly. A tiny p-value (commonly < 0.001) signals a broken randomisation and **invalidates the test**.

In [ ]:
counts = df.group.value_counts().reindex(["control", "treatment"])
chi2, p_srm = stats.chisquare(counts.values, f_exp=[counts.sum() / 2] * 2)
print(counts.to_dict())
print(f"SRM chi-square = {chi2:.3f}, p = {p_srm:.4f}")
print("SRM check:", "PASSED (no mismatch)" if p_srm > 0.001 else "FAILED - investigate randomisation!")

### 5.2 Covariate balance
Randomisation should balance pre-experiment characteristics such as device mix.

In [ ]:
ct_dev = pd.crosstab(df.group, df.device)
chi2, p_bal, dof, _ = stats.chi2_contingency(ct_dev)
print(ct_dev, "\n")
print(f"Device balance: chi2 = {chi2:.3f}, dof = {dof}, p = {p_bal:.4f}")
print("Balanced" if p_bal > ALPHA else "Imbalanced - consider stratified analysis")

### 5.3 A/A test: does our testing procedure control the false-positive rate?
We repeatedly split users from the **same** (control) population into two random halves. With α = 0.05, about 5% of tests should come out "significant" purely by chance.

In [ ]:
def two_prop_ztest(x1, n1, x2, n2, alpha=0.05):
    """Pooled two-proportion z-test (group 2 vs group 1). Returns dict with CI for difference."""
    p1, p2 = x1 / n1, x2 / n2
    p_pool = (x1 + x2) / (n1 + n2)
    se_pool = np.sqrt(p_pool * (1 - p_pool) * (1 / n1 + 1 / n2))
    z = (p2 - p1) / se_pool if se_pool > 0 else 0.0
    p_val = 2 * (1 - stats.norm.cdf(abs(z)))
    se_unpooled = np.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
    zc = stats.norm.ppf(1 - alpha / 2)
    diff = p2 - p1
    return dict(p1=p1, p2=p2, diff=diff, rel_lift=diff / p1 if p1 else np.nan,
                z=z, p_value=p_val, ci_low=diff - zc * se_unpooled, ci_high=diff + zc * se_unpooled)

rng = np.random.default_rng(7)
control_conv = df.loc[df.group == "control", "converted"].values
n_sims, false_pos = 2000, 0
for _ in range(n_sims):
    perm = rng.permutation(control_conv)
    half = len(perm) // 2
    r = two_prop_ztest(perm[:half].sum(), half, perm[half:].sum(), len(perm) - half)
    false_pos += r["p_value"] < ALPHA
print(f"A/A false-positive rate: {false_pos / n_sims:.3%} (expected ≈ 5%)")

### 5.4 The "peeking" problem
Checking results repeatedly and stopping at the first p < 0.05 inflates the Type I error. We simulate A/A tests with 10 interim looks.

In [ ]:
rng = np.random.default_rng(11)
n_sims, n_total, looks = 1500, 5000, 10
checkpoints = np.linspace(n_total / looks, n_total, looks).astype(int)
any_sig, final_sig = 0, 0
for _ in range(n_sims):
    a = rng.binomial(1, 0.10, n_total)
    b = rng.binomial(1, 0.10, n_total)          # same true rate -> no real effect
    sig_flags = []
    for k in checkpoints:
        r = two_prop_ztest(a[:k].sum(), k, b[:k].sum(), k)
        sig_flags.append(r["p_value"] < ALPHA)
    any_sig += any(sig_flags)
    final_sig += sig_flags[-1]
print(f"False positives, single look at the end : {final_sig / n_sims:.2%}")
print(f"False positives, stopping at ANY of {looks} looks: {any_sig / n_sims:.2%}")
print("-> Fix the sample size in advance (or use sequential methods / alpha spending).")

## 6. Primary metric: conversion rate

In [ ]:
c = df[df.group == "control"]; t = df[df.group == "treatment"]
x_c, n_c = c.converted.sum(), len(c)
x_t, n_t = t.converted.sum(), len(t)

# (a) Two-proportion z-test
res = two_prop_ztest(x_c, n_c, x_t, n_t)
print("=== Two-proportion z-test ===")
print(f"Control CR   : {res['p1']:.4%}")
print(f"Treatment CR : {res['p2']:.4%}")
print(f"Abs. diff    : {res['diff']*100:+.3f} pp   95% CI [{res['ci_low']*100:+.3f}, {res['ci_high']*100:+.3f}] pp")
print(f"Relative lift: {res['rel_lift']:+.2%}")
print(f"z = {res['z']:.3f}, p = {res['p_value']:.2e}")

# (b) Chi-square test of independence (no continuity correction -> equals z^2)
table = np.array([[x_c, n_c - x_c], [x_t, n_t - x_t]])
chi2, p_chi, dof, exp = stats.chi2_contingency(table, correction=False)
print(f"\n=== Chi-square ===\nchi2 = {chi2:.3f} (= z^2 = {res['z']**2:.3f}), p = {p_chi:.2e}")

# (c) Fisher's exact test
odds, p_fisher = stats.fisher_exact(table)
print(f"\n=== Fisher's exact ===\nodds ratio = {odds:.3f}, p = {p_fisher:.2e}")

# (d) Effect size: Cohen's h
h = 2 * (np.arcsin(np.sqrt(res["p2"])) - np.arcsin(np.sqrt(res["p1"])))
print(f"\nCohen's h = {h:.3f}  (0.2 small, 0.5 medium, 0.8 large)")

In [ ]:
# Visualise the difference and its confidence interval
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))

se = np.sqrt(res["p1"] * (1 - res["p1"]) / n_c + res["p2"] * (1 - res["p2"]) / n_t)
xs = np.linspace(res["diff"] - 4 * se, res["diff"] + 4 * se, 400)
ax[0].plot(xs * 100, stats.norm.pdf(xs, res["diff"], se))
ax[0].axvline(0, color="red", ls="--", label="H0: no difference")
ax[0].axvline(res["ci_low"] * 100, color="gray", ls=":"); ax[0].axvline(res["ci_high"] * 100, color="gray", ls=":")
ax[0].set_title("Sampling distribution of the difference (pp)"); ax[0].legend()

ax[1].errorbar(0, res["diff"] * 100, yerr=[[res["diff"] * 100 - res["ci_low"] * 100], [res["ci_high"] * 100 - res["diff"] * 100]],
               fmt="o", capsize=8)
ax[1].axhline(0, color="red", ls="--"); ax[1].set_xticks([]); ax[1].set_ylabel("Δ conversion (pp)")
ax[1].set_title("Lift with 95% CI")
plt.tight_layout(); plt.show()

## 7. Secondary metrics

### 7.1 Session duration — check assumptions first

In [ ]:
a = c.session_sec.values; b = t.session_sec.values

# Normality (Shapiro-Wilk on a subsample; D'Agostino-Pearson on all data)
print("Shapiro (control, n=2000):", stats.shapiro(np.random.default_rng(1).choice(a, 2000, replace=False)))
print("D'Agostino-Pearson control :", stats.normaltest(a))
print("Skewness control / treatment:", round(stats.skew(a), 3), round(stats.skew(b), 3))

# Equal variance
print("Levene test:", stats.levene(a, b))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
stats.probplot(a, dist="norm", plot=ax[0]); ax[0].set_title("Q-Q plot: control session duration")
stats.probplot(np.log(a), dist="norm", plot=ax[1]); ax[1].set_title("Q-Q plot: log(session duration)")
plt.tight_layout(); plt.show()

In [ ]:
def cohens_d(x, y):
    nx, ny = len(x), len(y)
    sp = np.sqrt(((nx - 1) * x.var(ddof=1) + (ny - 1) * y.var(ddof=1)) / (nx + ny - 2))
    return (y.mean() - x.mean()) / sp

def bootstrap_diff_ci(x, y, n_resamples=2000, seed=42):
    def diff(x_, y_, axis=-1):
        return np.mean(y_, axis=axis) - np.mean(x_, axis=axis)
    r = stats.bootstrap((x, y), diff, vectorized=True, n_resamples=n_resamples,
                        batch=200, method="percentile", random_state=seed)
    return r.confidence_interval.low, r.confidence_interval.high

# Welch t-test (robust to unequal variances; large n -> CLT makes it valid despite skew)
tt = stats.ttest_ind(b, a, equal_var=False)
ci = tt.confidence_interval()
print("=== Session duration ===")
print(f"Mean control = {a.mean():.1f}s, treatment = {b.mean():.1f}s, diff = {b.mean()-a.mean():+.1f}s")
print(f"Welch t = {tt.statistic:.3f}, p = {tt.pvalue:.2e}, 95% CI [{ci.low:.1f}, {ci.high:.1f}]")

# Non-parametric check
mw = stats.mannwhitneyu(b, a, alternative="two-sided")
print(f"Mann-Whitney U = {mw.statistic:.0f}, p = {mw.pvalue:.2e}")

lo, hi = bootstrap_diff_ci(a, b)
print(f"Bootstrap 95% CI for diff in means: [{lo:.1f}, {hi:.1f}]")
print(f"Cohen's d = {cohens_d(a, b):.3f}")
p_session = tt.pvalue

### 7.2 Revenue per user (zero-inflated, heavy-tailed)
Most users spend 0 and buyers' order values are log-normal, so the plain t-test is fragile here. We use Welch, Mann-Whitney and a bootstrap, and also analyse order value among buyers on the log scale.

In [ ]:
ra = c.revenue.values; rb = t.revenue.values
tt_r = stats.ttest_ind(rb, ra, equal_var=False)
mw_r = stats.mannwhitneyu(rb, ra, alternative="two-sided")
lo, hi = bootstrap_diff_ci(ra, rb)
print("=== Revenue per user (all users) ===")
print(f"Control = {ra.mean():.2f}, Treatment = {rb.mean():.2f}, diff = {rb.mean()-ra.mean():+.2f}")
print(f"Welch p = {tt_r.pvalue:.2e} | Mann-Whitney p = {mw_r.pvalue:.2e}")
print(f"Bootstrap 95% CI for diff: [{lo:.2f}, {hi:.2f}]")
p_revenue = tt_r.pvalue

# Average order value among buyers (log scale)
la = np.log(c.loc[c.converted == 1, "revenue"]); lb = np.log(t.loc[t.converted == 1, "revenue"])
tt_aov = stats.ttest_ind(lb, la, equal_var=False)
print("\n=== Order value among buyers (log scale) ===")
print(f"Geometric-mean ratio (T/C) = {np.exp(lb.mean() - la.mean()):.3f}, Welch p = {tt_aov.pvalue:.3f}")

### 7.3 Guardrail: bounce rate

In [ ]:
rb_ = two_prop_ztest(c.bounced.sum(), n_c, t.bounced.sum(), n_t)
print(f"Bounce rate: control {rb_['p1']:.2%} -> treatment {rb_['p2']:.2%}  "
      f"(Δ {rb_['diff']*100:+.2f} pp, 95% CI [{rb_['ci_low']*100:+.2f}, {rb_['ci_high']*100:+.2f}], p = {rb_['p_value']:.2e})")
p_bounce = rb_["p_value"]

## 8. Multiple-testing correction
We tested several metrics, so the chance of at least one false positive is above 5%. We apply **Bonferroni**, **Holm** and **Benjamini-Hochberg (FDR)** corrections.

In [ ]:
def adjust_pvalues(pvals, method="holm"):
    p = np.asarray(pvals, dtype=float); m = len(p)
    order = np.argsort(p); adj = np.empty(m)
    if method == "bonferroni":
        adj = np.minimum(p * m, 1.0)
    elif method == "holm":
        running = 0.0
        for rank, idx in enumerate(order):
            running = max(running, (m - rank) * p[idx])
            adj[idx] = min(running, 1.0)
    elif method == "bh":
        prev = 1.0
        for rank in range(m - 1, -1, -1):
            idx = order[rank]
            prev = min(prev, p[idx] * m / (rank + 1))
            adj[idx] = prev
    return adj

metrics = ["Conversion rate", "Session duration", "Revenue per user", "Bounce rate"]
raw_p = [res["p_value"], p_session, p_revenue, p_bounce]
mt = pd.DataFrame({"metric": metrics, "raw_p": raw_p})
for m_ in ["bonferroni", "holm", "bh"]:
    mt[m_] = adjust_pvalues(raw_p, m_)
mt["significant (Holm)"] = mt["holm"] < ALPHA
mt.style.format({"raw_p": "{:.2e}", "bonferroni": "{:.2e}", "holm": "{:.2e}", "bh": "{:.2e}"})

## 9. Segment analysis (device)
Heterogeneous effects are common in UI tests. Segment p-values must also be corrected, otherwise slicing the data finds "winners" by luck.

In [ ]:
rows = []
for dev, g in df.groupby("device"):
    gc, gt = g[g.group == "control"], g[g.group == "treatment"]
    r = two_prop_ztest(gc.converted.sum(), len(gc), gt.converted.sum(), len(gt))
    rows.append(dict(device=dev, n_control=len(gc), n_treatment=len(gt),
                     cr_control=r["p1"], cr_treatment=r["p2"], diff_pp=r["diff"] * 100,
                     ci_low_pp=r["ci_low"] * 100, ci_high_pp=r["ci_high"] * 100, p_value=r["p_value"]))
seg = pd.DataFrame(rows)
seg["p_holm"] = adjust_pvalues(seg.p_value, "holm")
seg["significant"] = seg.p_holm < ALPHA
display(seg.round(4))

plt.errorbar(seg.device, seg.diff_pp, yerr=[seg.diff_pp - seg.ci_low_pp, seg.ci_high_pp - seg.diff_pp], fmt="o", capsize=6)
plt.axhline(0, color="red", ls="--"); plt.ylabel("Δ conversion (pp)"); plt.title("Treatment effect by device (95% CI)")
plt.show()

# Formal interaction check: is the effect different across devices?
# Compare conversion tables (treatment-only vs control-only) via Breslow-Day-style intuition:
# a simple proxy is a chi-square test on the *difference* pattern using z-scores
z_vals = []
for dev, g in df.groupby("device"):
    gc, gt = g[g.group == "control"], g[g.group == "treatment"]
    r = two_prop_ztest(gc.converted.sum(), len(gc), gt.converted.sum(), len(gt))
    se_ = (r["ci_high"] - r["ci_low"]) / (2 * 1.96)
    z_vals.append((r["diff"], se_))
d_ = np.array([v[0] for v in z_vals]); s_ = np.array([v[1] for v in z_vals])
w = 1 / s_ ** 2
Q = np.sum(w * (d_ - np.sum(w * d_) / np.sum(w)) ** 2)           # Cochran's Q heterogeneity statistic
p_het = 1 - stats.chi2.cdf(Q, df=len(d_) - 1)
print(f"Heterogeneity of effect across devices: Cochran's Q = {Q:.3f}, p = {p_het:.4f}")

## 10. Bayesian view (Beta-Binomial)
Instead of a p-value, we estimate the **probability that Treatment beats Control** and the expected lift, using Beta(1,1) priors.

In [ ]:
rng = np.random.default_rng(3)
post_c = stats.beta(1 + x_c, 1 + n_c - x_c)
post_t = stats.beta(1 + x_t, 1 + n_t - x_t)
sc, st_ = post_c.rvs(200_000, random_state=rng), post_t.rvs(200_000, random_state=rng)

p_t_better = (st_ > sc).mean()
lift = (st_ - sc) / sc
print(f"P(Treatment > Control)       : {p_t_better:.4%}")
print(f"Expected relative lift       : {lift.mean():+.2%}")
print(f"95% credible interval (lift) : [{np.percentile(lift, 2.5):+.2%}, {np.percentile(lift, 97.5):+.2%}]")

xs = np.linspace(0.08, 0.14, 500)
plt.plot(xs, post_c.pdf(xs), label="Control"); plt.plot(xs, post_t.pdf(xs), label="Treatment")
plt.fill_between(xs, post_c.pdf(xs), alpha=0.2); plt.fill_between(xs, post_t.pdf(xs), alpha=0.2)
plt.title("Posterior distributions of conversion rate"); plt.xlabel("Conversion rate"); plt.legend(); plt.show()

## 11. Novelty effect / stability over time
A real effect should persist over the test window, rather than spike on day 1 and fade.

In [ ]:
daily = df.groupby(["date", "group"]).converted.mean().unstack()
daily["lift_pp"] = (daily["treatment"] - daily["control"]) * 100
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
daily[["control", "treatment"]].plot(ax=ax[0], marker="o"); ax[0].set_title("Daily conversion rate")
ax[1].bar(daily.index.strftime("%m-%d"), daily["lift_pp"]); ax[1].axhline(0, color="red", ls="--")
ax[1].set_title("Daily lift (pp)"); ax[1].tick_params(axis="x", rotation=60)
plt.tight_layout(); plt.show()

# Trend test: does the daily lift decay over time?
days = np.arange(len(daily))
slope = stats.linregress(days, daily["lift_pp"])
print(f"Trend in daily lift: slope = {slope.slope:+.4f} pp/day, p = {slope.pvalue:.3f}")
print("No significant decay -> no strong novelty effect" if slope.pvalue > ALPHA else "Lift is changing over time -> possible novelty effect")

## 12. Final report & decision

In [ ]:
primary_sig = mt.loc[0, "holm"] < ALPHA
guardrail_ok = not (rb_["diff"] > 0 and p_bounce < ALPHA)      # bounce must not get significantly worse
srm_ok = p_srm > 0.001

print("=" * 62)
print(" A/B TEST REPORT: New Product-Page UI vs Current UI")
print("=" * 62)
print(f"Users: control {n_c:,} | treatment {n_t:,}")
print(f"SRM check passed              : {srm_ok}")
print(f"Primary metric (conversion)   : {res['p1']:.2%} -> {res['p2']:.2%} "
      f"({res['rel_lift']:+.1%} relative, {res['diff']*100:+.2f} pp)")
print(f"  95% CI of absolute lift     : [{res['ci_low']*100:+.2f}, {res['ci_high']*100:+.2f}] pp")
print(f"  Holm-adjusted p-value       : {mt.loc[0, 'holm']:.2e}  -> significant: {primary_sig}")
print(f"Bayesian P(T > C)             : {p_t_better:.2%}")
print(f"Guardrail (bounce) acceptable : {guardrail_ok}")
print("-" * 62)

if srm_ok and primary_sig and guardrail_ok and res["ci_low"] > 0:
    decision = "LAUNCH the new UI (statistically significant, guardrails healthy)."
elif not srm_ok:
    decision = "DO NOT TRUST the result - fix the randomisation issue first."
elif not primary_sig:
    decision = "NO LAUNCH - no statistically significant improvement. Consider a longer test / new design."
else:
    decision = "INCONCLUSIVE - review guardrails and segment results before deciding."
print("DECISION:", decision)

mob = seg.loc[seg.device == "mobile"].iloc[0]
print(f"\nInsight: the lift is concentrated on mobile ({mob.diff_pp:+.2f} pp, Holm p = {mob.p_holm:.2e}).")

## 13. Conclusions & limitations

**Key takeaways**
- The test passed validity checks (SRM, covariate balance, A/A false-positive rate ≈ 5%).
- The new UI significantly improves conversion; the effect survives multiple-testing correction and agrees across frequentist (z-test, chi-square, Fisher) and Bayesian analyses.
- The effect is strongest on mobile, so a mobile-first rollout is justified.
- Peeking at results inflated false positives sharply, so the sample size must be fixed in advance.

**Limitations**
- Data is simulated; real data may include bots, repeat visitors, and interference between users.
- Short test windows can miss weekly seasonality and long-term effects.
- Revenue is heavy-tailed; very large orders can dominate means (consider winsorising or CUPED variance reduction).

**Possible extensions**
- CUPED variance reduction using pre-experiment behaviour
- Sequential testing (SPRT / alpha-spending) to allow safe early stopping
- Multi-armed bandits for continuous UI optimisation
- Logistic regression with covariates (statsmodels) as a robustness check